# What Is Machine Learning?

**Notebook 1 of 6.** The Palmer penguins dataset records 344 penguins from three islands
in Antarctica: their species, four body measurements, and their sex. For 11 of them the sex
is missing.

Those 11 are the interesting ones. Male and female penguins of these species look almost
alike, so determining the sex takes more than a tape measure and a scale. The four
measurements, on the other hand, take nothing else.

So here is the question for this repository: **can the measurements tell us the sex?**
Answering it is machine learning, and this notebook sets up the vocabulary before any
model is trained: what a model learns from, what it predicts, and what it is for.

First the library, then the data.

In [1]:
import pandas as pd

In [2]:
df_penguins = pd.read_csv("data/penguins.csv")
df_penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


## 1. From writing rules to learning them

In ordinary programming, **you write the rule**. You might decide that a penguin heavier
than 4,000 g is male, type an `if` statement, and the computer applies it.

In machine learning, **the computer finds the rule**. You hand it examples together with
the right answers, and an algorithm works out a rule that reproduces those answers as well
as it can.

| | You supply | The computer produces |
| :-- | :-- | :-- |
| Programming | data and a rule | answers |
| Machine learning | data and answers | a rule |

The rule the computer finds is called a **model**. Learning from examples that carry the
answer is **supervised learning**: the answers supervise the learning, like a teacher
marking practice questions. The answer attached to each example is its **label**.

## 2. Labeled and unlabeled penguins

A supervised model can only learn from rows that carry a label. How many do?

In [3]:
df_penguins["sex"].value_counts(dropna=False)

sex
MALE      168
FEMALE    165
NaN        11
Name: count, dtype: int64

333 penguins are labeled and 11 are not. The two groups have opposite jobs:

- The **labeled** penguins are the ones a model learns from.
- The **unlabeled** penguins are the reason to build a model at all. Their sex is exactly
  what nobody knows, and a model is only useful if it can fill that in.

In [4]:
labeled = df_penguins.dropna(subset=["sex"])
unlabeled = df_penguins[df_penguins["sex"].isna()]

print("labeled:  ", len(labeled))
print("unlabeled:", len(unlabeled))

labeled:   333
unlabeled: 11


In [5]:
unlabeled

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
8,Adelie,Torgersen,34.1,18.1,193.0,3475.0,NaN
9,Adelie,Torgersen,42.0,20.2,190.0,4250.0,NaN
10,Adelie,Torgersen,37.8,17.1,186.0,3300.0,NaN
11,Adelie,Torgersen,37.8,17.3,180.0,3700.0,NaN
47,Adelie,Dream,37.5,18.9,179.0,2975.0,NaN
246,Gentoo,Biscoe,44.5,14.3,216.0,4100.0,NaN
286,Gentoo,Biscoe,46.2,14.4,214.0,4650.0,NaN
324,Gentoo,Biscoe,47.3,13.8,216.0,4725.0,NaN
336,Gentoo,Biscoe,44.5,15.7,217.0,4875.0,NaN


> **Exercise:**
>
> A model that predicts sex from the four measurements needs all four of them. How many of
> the unlabeled penguins could it be used on?

<details><summary>
Click here for a hint...
</summary>

`dropna()` without arguments drops every row with any missing value. `len()` then counts
what is left.
</details>

In [19]:
# your code: unlabeled penguins with all four measurements

# Define columns name ### 
df_penguins.columns


labeled = df_penguins.dropna(subset=["bill_length_mm"])
unlabeled = df_penguins[df_penguins["bill_length_mm"].isna()]

print("labeled:  ", len(labeled))
print("unlabeled:", len(unlabeled))

print(f"labeled data is : \n { labeled } \n ")




labeled:   342
unlabeled: 2
labeled data is : 
     species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0    Adelie  Torgersen            39.1           18.7              181.0   
1    Adelie  Torgersen            39.5           17.4              186.0   
2    Adelie  Torgersen            40.3           18.0              195.0   
4    Adelie  Torgersen            36.7           19.3              193.0   
5    Adelie  Torgersen            39.3           20.6              190.0   
..      ...        ...             ...            ...                ...   
338  Gentoo     Biscoe            47.2           13.7              214.0   
340  Gentoo     Biscoe            46.8           14.3              215.0   
341  Gentoo     Biscoe            50.4           15.7              222.0   
342  Gentoo     Biscoe            45.2           14.8              212.0   
343  Gentoo     Biscoe            49.9           16.1              213.0   

     body_mass_g     sex  
0         37

## 3. Features and target

Every supervised problem splits the table into two parts.

- The **target** is the column you want to predict. By convention it is called `y`.
- The **features** are the columns the model is allowed to look at to make the
  prediction. By convention they are called `X`, with a capital letter because they form
  a table rather than a single column.

Here the target is `sex`, and the features are everything else.

In [20]:
y = labeled["sex"]
X = labeled.drop(columns="sex")

print("X:", X.shape)
print("y:", y.shape)

X: (342, 6)
y: (342,)


`X` has the same 333 rows as `y`, and one column fewer than the full table. The missing
column is the whole point:

In [ ]:
"sex" in X.columns

False

**Why the target must never be among the features.** A model that can see `sex` among its
features learns the shortest possible rule: copy that column. It scores perfectly on the
labeled penguins and is useless on the unlabeled ones, because for them that column is
empty. The one prediction you wanted, it cannot make.

This sounds too obvious to go wrong, but the target often sneaks back in disguised: as a
copy under another name, or as a column computed from it. Keep the check above as a habit.

> **Exercise:**
>
> A second question the same data can answer: **which species is a penguin?** Separate the
> labeled penguins into `X_species` and `y_species` for that question, and check that the
> target is not among the features.

In [23]:
# your code: features and target for predicting species
df_penguins.columns

# Separate rows with known and missing species
labeled = df_penguins.dropna(subset=["species"])
unlabeled = df_penguins[df_penguins["species"].isna()]

print("labeled:  ", len(labeled))
print("unlabeled:", len(unlabeled))

# Target and features
y = labeled["species"]
X = labeled.drop(columns="species")

print("X:", X.shape)
print("y:", y.shape)

# Should return False: the target must not be a feature
"species" in X.columns

labeled:   344
unlabeled: 0
X: (344, 6)
y: (344,)


False

## 4. Classification or regression

What the target looks like decides what kind of problem you have.

- **Classification:** the target is a **category**. Sex has two classes, so predicting it
  is **binary** classification. Species has three, which makes it **multiclass**.
- **Regression:** the target is a **number on a continuous scale**, such as a body mass in
  grams. The prediction can land anywhere between the values seen before.

The data types and the number of distinct values are a good first look:

In [24]:
pd.DataFrame({"dtype": df_penguins.dtypes, "distinct values": df_penguins.nunique()})

,dtype,distinct values
species,str,3
island,str,3
bill_length_mm,float64,164
bill_depth_mm,float64,80
flipper_length_mm,float64,55
body_mass_g,float64,94
sex,str,2


**Look at the meaning, not only the type.** A column stored as numbers can still be a
category. Passenger class on a ship is stored as 1, 2 and 3, and whether someone survived
as 0 and 1, but neither is a quantity: class 2 is not "twice" class 1. Both would be
classification targets.

> **Exercise:**
>
> No code needed. For each target below, decide whether predicting it is classification or
> regression, and for classification whether it is binary or multiclass.
>
> 1. `sex`
> 2. `species`
> 3. `body_mass_g`
> 4. `island`
> 5. `flipper_length_mm`

Your answers here:

<details><summary>
Click here to compare your answers...
</summary>

1. **Classification, binary.** Two classes, female and male.
2. **Classification, multiclass.** Three classes.
3. **Regression.** A mass in grams, on a continuous scale.
4. **Classification, multiclass.** Three islands. Predicting where a penguin was measured
   is an odd question, but a valid one.
5. **Regression.** Flipper length is recorded in whole millimeters, so it has only 55
   distinct values, but it is still a quantity: 200 mm really is longer than 190 mm.
</details>

## 5. What a feature is allowed to be

"Everything except the target" was a convenient choice above, not a rule. The actual rule
is about time:

> **A feature must be known at the moment the prediction is made.**

For the penguins that is easy. Species, island and the four measurements are all taken in
the field with a tape measure and a scale. But imagine the table also had a column
`sexing_batch`, the number of the batch in which a penguin's sex was determined. It
describes the penguin, and it may even predict sex well, if males and females happened to
be processed separately. It is still not a feature: at the moment of prediction the sex
has not been determined, so the column does not exist yet.

A feature that is only available *because* the answer is already known is called **target
leakage**. Notebook 05 is about it.

> **Exercise:**
>
> No code needed. A hospital wants to predict, at the moment a patient is discharged,
> whether they will be readmitted within 30 days. Which of these columns may be features?
>
> 1. The patient's age
> 2. The number of hospital stays in the previous year
> 3. The diagnosis at admission
> 4. The length of the patient's next hospital stay
> 5. Whether a follow-up appointment was booked before discharge

Your answers here:

<details><summary>
Click here to compare your answers...
</summary>

1. **Yes.** Known at discharge.
2. **Yes.** It describes the past.
3. **Yes.** Recorded when the patient arrived, before discharge.
4. **No.** It only exists if the patient is readmitted, so it contains the answer. A model
   trained with it would look excellent and fail on every new patient, for whom the column
   is always empty.
5. **Yes,** provided it really was booked before discharge. The same column filled in a
   week later would be a different matter: the rule is about *when* a value is known, not
   what it is called.
</details>

## 6. Generalization: the point of it all

Go back to the 333 labeled penguins. **A model that predicts their sex is not worth
anything in itself**, since we already know their sex. The only penguins a model is ever
used on are penguins it has **not** seen: the 11 unlabeled ones, and every penguin measured
in the future.

A model that performs well on data it has never seen is said to **generalize**. That is
the property you want, and the only one that matters in the end.

Which raises a practical problem. To check whether a model generalizes, you need penguins
it has not seen **but whose sex you know**, and every penguin with a known sex is one we
would like to learn from. Notebook 02 solves this by deliberately holding some back.

## 7. The workflow

Every supervised problem in this program follows the same steps. This repository walks
them in order:

```
question            can the measurements tell the sex?     01
   |
features, target    X and y, and what X may contain         01
   |
split               hold back a test set                    02
   |
baseline            the score any model has to beat         02
   |
model               logistic regression                     03, 04
   |
evaluate            on the test set, once                   02, 04
   |
check for leaks     features that know the answer           05
   |
all of it again     on Titanic passengers                   06
```

## Summary

- **Supervised learning** finds a rule from examples that carry the answer, their
  **labels**. The rule it finds is the **model**.
- The **target** `y` is what you predict, the **features** `X` are what the model sees,
  and **the target never appears among the features**.
- A categorical target makes a **classification** problem, binary or multiclass; a
  continuous one makes **regression**. Judge by meaning, not by data type.
- A feature must be **known at the moment of prediction**. One that is only known because
  the answer is, is **target leakage**.
- What a model is for is data it has not seen. Doing well there is **generalization**.

Next, in `02_split_and_baseline.ipynb`: hold some penguins back, and find out what score a
model has to beat before it counts as having learned anything.